In [ ]:
# Prerequisites
%pip install bitsandbytes peft  # for Memory optimization (QLoRA)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.1 MB/s eta 0:00:00:00:0100:01


In [8]:
# Dependencies

# Memory optimization
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import bitsandbytes as bnb
from peft import PeftModel, LoraConfig, get_peft_model

# Rest
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, get_linear_schedule_with_warmup
from datasets import Dataset, load_dataset

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import torch.nn.functional as F

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import time
import datetime

# For cleaner, encapsulated code between cells
class Local:
    def __init__(self, **kwords):
        self.__dict__ = kwords

In [ ]:
# GPU
_ = Local()
if torch.cuda.is_available():
    _.gpu_stats = torch.cuda.get_device_properties(0)
    print(f"GPU = {_.gpu_stats.name}.")
    _.gpu_memory = round(_.gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
    print(f"GPU Memory = {_.gpu_memory} GB")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device : {device}")

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.eos_token_id

Using device : cpu


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [14]:
# Load/Download Qwen
_ = Local()

_.bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,              # Enable 4-bit quantization
    bnb_4bit_quant_type="nf4",      # NormalFloat4: best for neural net weights
    bnb_4bit_compute_dtype=torch.bfloat16, # For GPU T4
    bnb_4bit_use_double_quant=True, # Nested quantization for extra memory savings
)

_.lora_config = LoraConfig(
    r=16,                           # Rank (16 is standard for 1.5B models)
    lora_alpha=32,                  # Scaling factor (2 * r)
    lora_dropout=0.05,              # Dropout for regularization
    bias="none",                    # Do not train bias vectors
    task_type="CAUSAL_LM",          # Causal Language Modeling
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",  # Attention layers
        "gate_proj", "up_proj", "down_proj"      # MLP layers
    ],
    inference_mode=False,           # Must be False for training
)

def get_qwen():
    fin = AutoModelForCausalLM.from_pretrained(
        "Qwen/Qwen2.5-1.5B-Instruct",
        quantization_config=_.bnb_config,
        dtype=torch.bfloat16,       # Match compute dtype
    ).to(device)
    fin.config.pad_token_id = tokenizer.pad_token_id
    return fin

model = get_qwen()
model = get_peft_model(model, _.lora_config)
model.print_trainable_parameters()

ref_model = get_qwen()

model

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/default/ops.py:223: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cpu/ops.py:36: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora